In [0]:
%run "/Workspace/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Workspace/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/DynamicColumnMapper"

In [0]:
%run "/Workspace/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/BronzeLoader"

In [0]:
bronze_table_name = f"{catalog_name}.{bronze_schema}.orders"
source_volumne = get_volume_path("raw") + "/customer_360_project_data/orders.csv"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("order_status", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
%skip
%sql
DROP TABLE IF EXISTS customer_360.bronze.orders;
DELETE FROM customer_360.gold.pipeline_watermark WHERE table_name = 'customer_360.bronze.orders';

In [0]:
df = spark.read.csv(source_volumne, header=True, schema=orders_schema)

final_df = add_ingestion_metadata(df, source_volumne)

try:
    load_to_bronze("orders", source_volumne, df)
    # display(spark.read.table(bronze_table_name))
except Exception as e:
    print(e)
